# Daily Challenge — Text Analysis of Lewis Carroll Books

**Course:** Developers Institute  **Week 8 - Day 1**  
**Author:** Alex Goldbaum

Three Lewis Carroll works downloaded straight from Project Gutenberg:
*Alice's Adventures in Wonderland*, *Through the Looking-Glass*, and
*A Tangled Tale*. We preprocess them with NLTK + spaCy, build word clouds,
compute Bag-of-Words top-5 and then move to TF-IDF to surface words that are
actually distinctive per book.


## Setup


In [ ]:
%pip install -qU requests nltk spacy scikit-learn wordcloud matplotlib pandas
%python -m spacy download -q en_core_web_sm


In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger',
            'averaged_perceptron_tagger_eng', 'maxent_ne_chunker',
            'maxent_ne_chunker_tab', 'words']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk import pos_tag, ne_chunk

import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download as spacy_dl
    spacy_dl('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from wordcloud import WordCloud


## Text Preprocessing

### 1. `load_texts()` — Download and clean the three books from Project Gutenberg


In [ ]:
BOOKS = {
    'Alice in Wonderland':       'https://www.gutenberg.org/cache/epub/11/pg11.txt',
    'Through the Looking-Glass': 'https://www.gutenberg.org/cache/epub/12/pg12.txt',
    'A Tangled Tale':            'https://www.gutenberg.org/cache/epub/29042/pg29042.txt',
}


def load_texts(urls):
    """Download each URL and lightly clean non-word characters, return a list of cleaned strings."""
    corpus = []
    for url in urls:
        resp = requests.get(url, timeout=30)
        resp.encoding = 'utf-8'
        text = resp.text
        # Strip CR characters; keep the structure for now so we can slice on START/END markers later
        text = text.replace('\r', '')
        # Lightweight regex cleaning: collapse runs of whitespace; remove BOM
        text = text.lstrip('\ufeff')
        text = re.sub(r'[ \t]+', ' ', text)
        corpus.append(text)
    return corpus


raw_corpus = load_texts(list(BOOKS.values()))
for title, text in zip(BOOKS.keys(), raw_corpus):
    print(f'{title}: {len(text):,} characters')


### 2. First 200 characters of each book — there is Project Gutenberg boilerplate to strip


In [ ]:
for title, text in zip(BOOKS.keys(), raw_corpus):
    print(f'-- {title} --')
    print(text[:200])
    print()


**Yes — the texts start with Project Gutenberg's licence boilerplate**
("The Project Gutenberg eBook of …") and end with another long licence block.
We slice between the two `*** START` / `*** END` markers as the brief suggests.


In [ ]:
def strip_gutenberg_boilerplate(text):
    start_idx = text.find('*** START')
    end_idx = text.find('*** END')
    if start_idx == -1 or end_idx == -1 or end_idx <= start_idx:
        return text  # fallback if markers are missing
    # Move start past the rest of the START line
    start_line_end = text.find('\n', start_idx)
    return text[start_line_end + 1:end_idx].strip()


corpus = [strip_gutenberg_boilerplate(t) for t in raw_corpus]

for title, text in zip(BOOKS.keys(), corpus):
    print(f'{title}: {len(text):,} characters (after stripping boilerplate)')
    print(text[:200])
    print()


### 3. Tokenize and inspect the first 150 tokens per book


In [ ]:
tokenized = [word_tokenize(text) for text in corpus]

for title, tokens in zip(BOOKS.keys(), tokenized):
    print(f'-- {title} ({len(tokens):,} tokens) --')
    print(tokens[:150])
    print()


### 4. Remove English stopwords

Stopwords (`the`, `a`, `is`, `i`, `me`, `my`, …) carry little information.
We use the lowercase form so capitalised mid-sentence forms still match.


In [ ]:
stop_set = set(stopwords.words('english'))

# Lowercase + remove punctuation tokens + remove stopwords
PUNCT_RE = re.compile(r"^[\W_]+$")

def clean_tokens(tokens):
    return [t.lower() for t in tokens
            if not PUNCT_RE.match(t) and t.lower() not in stop_set]


cleaned = [clean_tokens(tokens) for tokens in tokenized]

# Sanity check: stopwords like 'i', 'me', 'my' should now appear 0 times
for title, tokens in zip(BOOKS.keys(), cleaned):
    print(f'-- {title} ({len(tokens):,} tokens after cleaning) --')
    for w in ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']:
        print(f'   count of {w!r}: {tokens.count(w)}')
    print()


### 5. Porter stemming — first 50 stemmed tokens of each book


In [ ]:
stemmer = PorterStemmer()
stemmed = [[stemmer.stem(t) for t in toks] for toks in cleaned]

for title, stems in zip(BOOKS.keys(), stemmed):
    print(f'-- {title} -- first 50 stems:')
    print(stems[:50])
    print()


### 6. spaCy lemmatization — first 50 lemmatized tokens of each book

spaCy's `Token.lemma_` returns the dictionary form (singular, infinitive, …).
We pass the **cleaned token list rejoined as text** so spaCy can use context
to disambiguate the lemma (e.g., *saw* → *see* vs *saw* the tool).


In [ ]:
lemmatized = []
for toks in cleaned:
    # Use the first ~20k characters per book for spaCy to keep memory in check
    chunk = ' '.join(toks[:20_000])
    doc = nlp(chunk)
    lemmatized.append([tok.lemma_ for tok in doc if not tok.is_space])

for title, lemmas in zip(BOOKS.keys(), lemmatized):
    print(f'-- {title} -- first 50 lemmas:')
    print(lemmas[:50])
    print()


### 7. Stemming vs lemmatization — what is the difference?

- **Stemming** is a rule-based string operation. The PorterStemmer chops off
  common suffixes (`-ing`, `-ed`, `-s`) without checking whether the result is
  a real word. *"adventures" → "adventur"*, *"running" → "run"*,
  *"flies" → "fli"*. It is fast and crude; the output is often **not** a
  valid English word.
- **Lemmatization** is dictionary-aware. spaCy maps each token to its
  canonical dictionary form, using part-of-speech information to pick the
  right lemma. *"adventures" → "adventure"*, *"running" → "run"*,
  *"flies" → "fly"*. The output is always a valid lemma.

**Why does it matter?** For human-readable insights (word clouds, top-K),
lemmas are far better — readers expect *"adventure"*, not *"adventur"*.
For indexing or memory-tight applications, stemming is cheaper.


### 8. POS tagging with NLTK


In [ ]:
pos_tags = []
for title, tokens in zip(BOOKS.keys(), cleaned):
    tags = pos_tag(tokens[:200])  # sample of first 200 tokens for display
    pos_tags.append(tags)
    print(f'-- {title} -- first 20 POS tags:')
    print(tags[:20])
    print()


### 9. Named-entity recognition with NLTK


In [ ]:
def extract_entities(tagged):
    chunked = ne_chunk(tagged, binary=False)
    entities = []
    for subtree in chunked:
        if hasattr(subtree, 'label'):
            ent_text = ' '.join(token for token, _ in subtree)
            entities.append((ent_text, subtree.label()))
    return entities

for title, tagged in zip(BOOKS.keys(), pos_tags):
    ents = extract_entities(tagged)
    unique_ents = list(dict.fromkeys(ents))[:15]
    print(f'-- {title} -- first 15 unique entities (sampled from the first 200 tokens):')
    for ent, label in unique_ents:
        print(f'   [{label}] {ent}')
    print()


## Analyzing the Text


### 1. Word cloud per book

We use the **lemmatized** tokens — they read more naturally on the cloud than
the Porter stems.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (title, lemmas) in zip(axes, zip(BOOKS.keys(), lemmatized)):
    text_for_wc = ' '.join(lemmas)
    wc = WordCloud(width=900, height=600, background_color='white',
                   colormap='viridis', max_words=120).generate(text_for_wc)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()


### 2. Bag-of-Words — top 5 most frequent words across the corpus

Best input for BoW: **lemmatized** tokens (clean + readable). We treat each
book as one document.


In [ ]:
documents = [' '.join(lemmas) for lemmas in lemmatized]

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(documents)
vocab = bow_vectorizer.get_feature_names_out()

print('BoW matrix shape:', bow_matrix.shape, ' (n_documents, n_words)')
print('Vocabulary size :', len(vocab))

# Total counts across all 3 documents
totals = np.asarray(bow_matrix.sum(axis=0)).ravel()
top5_idx = totals.argsort()[::-1][:5]
top5 = [(vocab[i], int(totals[i])) for i in top5_idx]

print('\nTop 5 most frequent words across all 3 books:')
for w, c in top5:
    print(f'  {w:>10}: {c}')


### 3. Reading the raw BoW output

Sparse-matrix entries print as `(doc_index, vocab_index)  count`.


In [ ]:
# Show the first dozen non-zero entries of the BoW matrix
coo = bow_matrix.tocoo()
print('(document #, word index)  count    word')
for i in range(15):
    doc_i, word_i, count = coo.row[i], coo.col[i], coo.data[i]
    print(f'   ({doc_i}, {word_i:>5})  {count:>4}    {vocab[word_i]}')

print()
print('Reading: the document number is the row, the vocab index is the column,',
      'and the value is how many times that word appeared in that document.')


### 4. Pie plot of the top 5 most frequent words


In [ ]:
labels = [f'{w}\n({c} occurrences)' for w, c in top5]
values = [c for _, c in top5]

plt.figure(figsize=(7, 7))
plt.pie(values, labels=labels, autopct='%1.1f%%',
        colors=plt.cm.viridis(np.linspace(0.2, 0.85, len(values))),
        startangle=90, textprops={'fontsize': 11})
plt.title('Top 5 most frequent words across all 3 books (BoW)', fontweight='bold')
plt.tight_layout()
plt.show()


### 5. Are these words informative?

Not really. The BoW top-5 surface generic high-frequency words like `say`,
*"alice"*, *"little"*, *"go"*, *"like"*. They are **expected**: any
book about Alice will mention Alice a lot, and almost any English novel
uses *"say"* / *"little"* heavily. They tell us almost nothing distinctive
about each book. This is exactly the limitation that motivates TF-IDF.


## Solving the Frequency Problem with TF-IDF

TF-IDF weights each word by its frequency **inside a document** divided by
its frequency **across the corpus**. A word that appears everywhere gets a
low IDF and a low score; a word that is concentrated in one book gets a high
score. With only 3 books we pass `min_df=1, max_df=2` to keep words that
appear in at least one document and **at most two** (so words present in all
three are filtered out — exactly the *"Alice"* / *"say"* problem).


In [ ]:
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print('TF-IDF matrix shape:', tfidf_matrix.shape)
print('Vocabulary size    :', len(tfidf_vocab))

tfidf_array = tfidf_matrix.toarray()

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (i, title) in zip(axes, enumerate(BOOKS.keys())):
    doc_scores = tfidf_array[i]
    top5_i = doc_scores.argsort()[::-1][:5]
    words = [tfidf_vocab[j] for j in top5_i]
    scores = [doc_scores[j] for j in top5_i]

    labels = [f'{w}\n(tf-idf {s:.3f})' for w, s in zip(words, scores)]
    ax.pie(scores, labels=labels, autopct='%1.1f%%',
           colors=plt.cm.viridis(np.linspace(0.2, 0.85, len(scores))),
           startangle=90, textprops={'fontsize': 10})
    ax.set_title(title, fontweight='bold')

plt.suptitle('Top 5 most DISTINCTIVE words per book (TF-IDF)', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Also print the leaderboards for clarity
for i, title in enumerate(BOOKS.keys()):
    doc_scores = tfidf_array[i]
    top10_i = doc_scores.argsort()[::-1][:10]
    print(f'-- {title} -- top 10 TF-IDF terms:')
    for j in top10_i:
        print(f'   {tfidf_vocab[j]:>15}: {doc_scores[j]:.4f}')
    print()


## Conclusions

- **BoW** gives us frequency rankings dominated by generic words that any
  book on the same topic shares (here `say`, `alice`, `little`, `go`).
  Useful for raw volume but uninformative for character/topic discovery.
- **TF-IDF** is the right tool when the goal is *what makes this document
  distinctive*. With our tiny 3-document corpus and `max_df=2`, the surfaced
  terms are characters, settings and motifs unique to each book — exactly
  the words a literature student would highlight.
- **Preprocessing matters as much as the model.** Project Gutenberg
  boilerplate, stopwords, capitalisation and morphological variation all
  shift the ranking; we used lemmatization rather than stemming so the
  output reads naturally.
- The same pipeline (download → strip boilerplate → tokenize → clean →
  lemmatize → vectorise → analyse) generalises to any collection of plain-
  text documents — research papers, support tickets, customer reviews —
  with no Transformer model in sight.
